# 03 — ViT + CIFAR PEFT comparison

This notebook connects the toy intuition from the first two notebooks to a **real pretrained vision model**.

It is still intentionally small and workshop-friendly.

## Methods compared
- linear probing
- partial finetuning
- visual prompt tuning
- a LoRA-style low-rank head update (minimal teaching version)

## Goal
Use the same dataset and checkpoint to get early intuition about:
- trainable parameter count
- convergence behavior
- quality vs complexity tradeoff
- what to use when


In [ ]:
# If running in Colab directly from GitHub, first clone the repo so `src/` is available:
# !git clone https://github.com/<your-user>/<your-repo>.git
# %cd <your-repo>
# !pip install -q torch torchvision matplotlib pandas tqdm

import os
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights

ROOT = Path.cwd()
if (ROOT / ".." / "src").exists():
    sys.path.append(str((ROOT / "..").resolve()))
elif (ROOT / "src").exists():
    sys.path.append(str(ROOT.resolve()))

from src.data import make_cifar10_loaders
from src.training import train_model, evaluate, count_trainable_parameters, freeze_module
from src.visualization import plot_history, summarize_results
from src.methods.prompt_tuning import PromptTunedClassifier
from src.methods.adapters import AdapterHeadClassifier


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Dataset

For workshop runtime, you can use a subset of CIFAR-10.  
Increase `subset_fraction` or `epochs` if you want stronger results.


In [ ]:
bundle = make_cifar10_loaders(
    batch_size=16,
    image_size=224,
    subset_fraction=0.05,   # raise this later if you want stronger results
)
bundle.num_classes, bundle.input_shape

## Backbone wrapper

Torchvision's ViT returns logits directly, so we wrap it to expose the frozen backbone features.


In [ ]:

class ViTFeatureBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        weights = ViT_B_16_Weights.DEFAULT
        self.vit = vit_b_16(weights=weights)
        self.feature_dim = self.vit.heads.head.in_features
        self.vit.heads = nn.Identity()

    def forward(self, x):
        return self.vit(x)


## Methods

We keep the comparison tight and readable.  
For a live workshop, it is better to compare **4 methods well** than 10 methods superficially.


In [ ]:

class LinearProbeClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        freeze_module(self.backbone)
        self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


class PartialFineTuneClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        freeze_module(self.backbone)

        # unfreeze final encoder block + norm as a small partial-finetuning baseline
        for p in self.backbone.vit.encoder.layers[-1].parameters():
            p.requires_grad = True
        for p in self.backbone.vit.encoder.ln.parameters():
            p.requires_grad = True

        self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


class LowRankHead(nn.Module):
    """A tiny LoRA-like low-rank classifier head for teaching purposes."""
    def __init__(self, backbone, feature_dim, num_classes, rank=8, alpha=16):
        super().__init__()
        self.backbone = backbone
        freeze_module(self.backbone)

        self.base = nn.Linear(feature_dim, num_classes)
        self.base.weight.requires_grad = False
        self.base.bias.requires_grad = False

        self.A = nn.Linear(feature_dim, rank, bias=False)
        self.B = nn.Linear(rank, num_classes, bias=False)
        self.scaling = alpha / rank

    def forward(self, x):
        feats = self.backbone(x)
        return self.base(feats) + self.B(self.A(feats)) * self.scaling


In [ ]:
def build_method(name: str):
    backbone = ViTFeatureBackbone()
    feat_dim = backbone.feature_dim
    nc = bundle.num_classes

    if name == "linear_probe":
        return LinearProbeClassifier(backbone, feat_dim, nc)
    if name == "partial_finetuning":
        return PartialFineTuneClassifier(backbone, feat_dim, nc)
    if name == "visual_prompt_tuning":
        return PromptTunedClassifier(backbone, feat_dim, nc, prompt_size=24)
    if name == "adapter_head":
        return AdapterHeadClassifier(backbone, feat_dim, nc, bottleneck_dim=64)
    if name == "low_rank_head":
        return LowRankHead(backbone, feat_dim, nc, rank=8, alpha=16)
    raise ValueError(f"Unknown method: {name}")

method_names = [
    "linear_probe",
    "partial_finetuning",
    "visual_prompt_tuning",
    "adapter_head",
    "low_rank_head",
]

param_preview = []
for name in method_names:
    model = build_method(name)
    param_preview.append({
        "method": name,
        "trainable_params": count_trainable_parameters(model),
    })
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

param_preview

## Train

Keep the epoch count small in the workshop.  
This notebook now trains methods **one at a time** so it is much less likely to run out of memory in Colab.

Participants can rerun one method with more epochs afterward.


In [ ]:
histories = {}
results = []

for name in method_names:
    print(f"\n=== Training {name} ===")
    model = build_method(name).to(device)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-3,
        weight_decay=1e-4,
    )
    history = train_model(
        model,
        bundle.train_loader,
        bundle.val_loader,
        optimizer=optimizer,
        epochs=2,
        device=device,
    )
    metrics = evaluate(model, bundle.val_loader, device=device)
    histories[name] = history
    results.append({
        "method": name,
        "trainable_params": count_trainable_parameters(model),
        "val_acc": metrics["acc"],
        "val_loss": metrics["loss"],
    })

    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results

In [ ]:
plot_history(histories)

In [ ]:
summary = summarize_results(results)
summary

## Discussion prompts

Use the table and curves to ask:

1. Which method gave the best performance for its trainable parameter budget?
2. Did the cheap baseline (linear probe) already do surprisingly well?
3. Did adding trainable capacity inside the computation help more than only changing the head?
4. Which method would you choose if:
   - you need the smallest per-task checkpoint?
   - you want a strong first PEFT baseline?
   - you have enough compute and want more flexibility?

## Rough heuristic

- **Linear probe**: start here when features may already be good enough
- **Visual prompt tuning**: attractive when you want minimal backbone disturbance
- **Adapters**: useful when you want modular hidden-state intervention
- **Low-rank / LoRA-style updates**: strong default PEFT baseline in many settings
- **Partial finetuning**: useful when you can afford a bit more adaptation capacity
